In [1]:
import numpy as np
import ngmix
import galsim
import matplotlib.pyplot as plt
from IPython.display import display, Math
from matplotlib.colors import LogNorm
import matplotlib.colors as colors
# from shearnet.methods.ngmix import _get_priors
from astropy.table import Table
import superbit_lensing.utils as utils
norm2 = colors.SymLogNorm(linthresh=1e-4,base=np.e)
cosmos_cat_fname = "/projects/mccleary_group/superbit/galsim_data/cosmos15_superbit2023_phot_shapes_with_sigma.csv"
cosmos_cat = Table.read(cosmos_cat_fname, format="csv")

def _get_priors(seed):

    # This bit is needed for ngmix v2.x.x
    # won't work for v1.x.x
    rng = np.random.RandomState(seed)

    # prior on ellipticity.  The details don't matter, as long
    # as it regularizes the fit.  This one is from Bernstein & Armstrong 2014

    g_sigma = 0.3
    g_prior = ngmix.priors.GPriorBA(g_sigma, rng=rng)

    # 2-d gaussian prior on the center
    # row and column center (relative to the center of the jacobian, which would be zero)
    # and the sigma of the gaussians

    # units same as jacobian, probably arcsec
    row, col = 0.0, 0.0
    row_sigma, col_sigma = 0.2, 0.2 # a bit smaller than pix size of SuperBIT
    cen_prior = ngmix.priors.CenPrior(row, col, row_sigma, col_sigma, rng=rng)

    # T prior.  This one is flat, but another uninformative you might
    # try is the two-sided error function (TwoSidedErf)

    Tminval = -1.0 # arcsec squared
    Tmaxval = 1000
    T_prior = ngmix.priors.FlatPrior(Tminval, Tmaxval, rng=rng)

    # similar for flux.  Make sure the bounds make sense for
    # your images

    Fminval = -1.e1
    Fmaxval = 1.e5
    F_prior = ngmix.priors.FlatPrior(Fminval, Fmaxval, rng=rng)

    # now make a joint prior.  This one takes priors
    # for each parameter separately
    priors = ngmix.joint_prior.PriorSimpleSep(
    cen_prior,
    g_prior,
    T_prior,
    F_prior)

    return priors

In [2]:
def make_data(rng, noise, shear_true):
    """
    simulate an exponential object with gaussian psf

    Parameters
    ----------
    rng: np.random.RandomState
        The random number generator
    noise: float
        Noise for the image
    shear: (g1, g2)
        The shear in each component

    Returns
    -------
    ngmix.Observation
    """

    psf_noise = 1.0e-6

    scale = 0.141

    psf_fwhm = 0.5
    gal_hlr = 0.5
    index = rng.randint(len(cosmos_cat))
    phi = cosmos_cat[index]['c10_sersic_fit_phi'] * galsim.radians
    q   = cosmos_cat[index]['c10_sersic_fit_q']   
    if q > 1.0:
        q = 1/q 
    
    npix_psf = 51
    npix = 64
    dy, dx = rng.uniform(low=-scale/2, high=scale/2, size=2)

    psf = galsim.Gaussian(fwhm=psf_fwhm)

    obj0 = galsim.Exponential(half_light_radius=gal_hlr).shear(q=q, beta=phi)
    objp = obj0.shear(g1=shear_true, g2=0.0).shift(dx=dx, dy=dy)
    objm = obj0.shear(g1=-shear_true, g2=0.0).shift(dx=dx, dy=dy)

    obj0_psf = galsim.Convolve(psf, obj0)
    objp_psf = galsim.Convolve(psf, objp)
    objm_psf = galsim.Convolve(psf, objm)

    psf_im = psf.drawImage(nx=npix_psf, ny=npix_psf, scale=scale).array
    im_0 = obj0_psf.drawImage(nx=npix, ny=npix, scale=scale).array
    im_p = objp_psf.drawImage(nx=npix, ny=npix, scale=scale).array
    im_m = objm_psf.drawImage(nx=npix, ny=npix, scale=scale).array

    psf_im += rng.normal(scale=psf_noise, size=psf_im.shape)
    im_noise = rng.normal(scale=noise, size=im_0.shape)
    im_0 += im_noise
    im_p += im_noise
    im_m += im_noise

    cen = (np.array(im_0.shape)-1.0)/2.0
    psf_cen = (np.array(psf_im.shape)-1.0)/2.0

    jacobian = ngmix.DiagonalJacobian(
        row=cen[0] + dy/scale, col=cen[1] + dx/scale, scale=scale,
    )
    psf_jacobian = ngmix.DiagonalJacobian(
        row=psf_cen[0], col=psf_cen[1], scale=scale,
    )

    wt = im_0*0 + 1.0/noise**2
    psf_wt = psf_im*0 + 1.0/psf_noise**2

    psf_obs = ngmix.Observation(
        psf_im,
        weight=psf_wt,
        jacobian=psf_jacobian,
    )

    obs0 = ngmix.Observation(im_0, weight=wt, jacobian=jacobian, psf=psf_obs)
    obsp = ngmix.Observation(im_p, weight=wt, jacobian=jacobian, psf=psf_obs)
    obsm = ngmix.Observation(im_m, weight=wt, jacobian=jacobian, psf=psf_obs)

    return obs0, obsp, obsm


def progress(total, miniters=1):
    last_print_n = 0
    last_printed_len = 0
    sl = str(len(str(total)))
    mf = '%'+sl+'d/%'+sl+'d %3d%%'
    for i in range(total):
        yield i

        num = i+1
        if i == 0 or num == total or num - last_print_n >= miniters:
            meter = mf % (num, total, 100*float(num) / total)
            nspace = max(last_printed_len-len(meter), 0)

            print('\r'+meter+' '*nspace, flush=True, end='')
            last_printed_len = len(meter)
            if i > 0:
                last_print_n = num

    print(flush=True)


def make_struct(res, obs, shear_type):
    """
    make the data structure

    Parameters
    ----------
    res: dict
        With keys 's2n', 'e', and 'T'
    obs: ngmix.Observation
        The observation for this shear type
    shear_type: str
        The shear type

    Returns
    -------
    1-element array with fields
    """
    dt = [
        ('flags', 'i4'),
        ('shear_type', 'U7'),
        ('s2n', 'f8'),
        ('g', 'f8', 2),
        ('T', 'f8'),
        ('Tpsf', 'f8'),
        ('gpsf', 'f8', 2)
    ]
    data = np.zeros(1, dtype=dt)
    data['shear_type'] = shear_type
    data['flags'] = res['flags']
    if res['flags'] == 0:
        data['s2n'] = res['s2n']
        # for moments we are actually measureing e, the elliptity
        try:
            data['g'] = res['e']
        except KeyError:
            data['g'] = res['g']
        data['T'] = res['T']
    else:
        data['s2n'] = np.nan
        data['g'] = np.nan
        data['T'] = np.nan
        data['Tpsf'] = np.nan

    # we only have one epoch and band, so we can get the psf T from the
    # observation rather than averaging over epochs/bands
    admom_dict = utils.get_admoms_ngmix_fit(obs.psf, reduced=True)
    if admom_dict['flags'] == 0:
        g1psf, g2psf, Tpsf = admom_dict['e1'], admom_dict['e2'], admom_dict['T']
    else:
        g1psf, g2psf, Tpsf = np.nan, np.nan, np.nan
    data['Tpsf'] = Tpsf
    data['gpsf'] = np.array([g1psf, g2psf])
    #data['Tpsf'] = obs.psf.meta['result']['T']

    return data

def process_obs(obs, boot):
    resdict, obsdict = boot.go(obs)
    dlist = [make_struct(res=sres, obs=obsdict[stype], shear_type=stype) for stype, sres in resdict.items()]
    return np.hstack(dlist)

def shear_data_to_table(data_list, mcal_shear=0.01):
    rows = []

    for arr in data_list:
        row = {}
        g_store = {}

        # single combined flag
        row["flag"] = 0 if np.all(arr["flags"] == 0) else 1

        for rec in arr:
            stype = rec["shear_type"]

            for name in arr.dtype.names:
                if name in ("shear_type", "flags"):
                    continue

                val = rec[name]

                # keep g as a 2-vector column
                if name == "g":
                    g_store[stype] = np.array(val, dtype=float)
                    row[f"g_{stype}"] = np.array(val, dtype=float)
                    continue

                # everything else is scalar here (s2n, T, Tpsf, etc.)
                row[f"{name}_{stype}"] = val

        dg = 2 * mcal_shear

        # r11 from 1p/1m (g1 component)
        if "1p" in g_store and "1m" in g_store:
            row["r11"] = (g_store["1p"][0] - g_store["1m"][0]) / dg

        # r22 only if 2p/2m exist (g2 component)
        if "2p" in g_store and "2m" in g_store:
            row["r22"] = (g_store["2p"][1] - g_store["2m"][1]) / dg

        rows.append(row)

    return Table(rows)

def jackknife_mc(tab, shear_true, njac=20):
    """
    Block jackknife over rows of `tab` into `njac` subsamples.
    Returns m_mean, m_err, c_mean, c_err, plus useful diagnostics.
    """
    N = len(tab)
    if njac < 2:
        raise ValueError("njac must be >= 2")
    if njac > N:
        raise ValueError(f"njac ({njac}) cannot exceed number of rows ({N})")
    g_arr = np.asarray(tab["g_noshear"])   # (N,2)
    R_arr = np.asarray(tab["r11"])         # (N,)

    indices = np.arange(N)
    chunks = np.array_split(indices, njac)
    m_jack = []
    c_jack = []
    for chunk in chunks:
        mask = np.ones(N, dtype=bool)
        mask[chunk] = False
        g_mean_chunk = np.mean(g_arr[mask], axis=0)
        R_mean_chunk = np.mean(R_arr[mask])
        shear_chunk = g_mean_chunk/R_mean_chunk
        m_chunk = shear_chunk[0]/shear_true[0] - 1
        c_chunk = shear_chunk[1]
        m_jack.append(m_chunk)
        c_jack.append(c_chunk)

    m_jack = np.array(m_jack)
    c_jack = np.array(c_jack)

    m_mean = np.mean(m_jack)
    c_mean = np.mean(c_jack)
    m_err = np.sqrt((njac - 1) * np.mean((m_jack - m_mean)**2))
    c_err = np.sqrt((njac - 1) * np.mean((c_jack - c_mean)**2))

    return m_mean, m_err, c_mean, c_err, m_jack, c_jack

def jackknife_mc_v2(tab_p, tab_m, shear_true, njac=20):
    """
    Block jackknife over rows of `tab` into `njac` subsamples.
    Returns m_mean, m_err, c_mean, c_err, plus useful diagnostics.
    """
    N = len(tab_p)
    if njac < 2:
        raise ValueError("njac must be >= 2")
    if njac > N:
        raise ValueError(f"njac ({njac}) cannot exceed number of rows ({N})")
    g_arr_p = np.asarray(tab_p["g_noshear"])   # (N,2)
    R_arr_p = np.asarray(tab_p["r11"])         # (N,)

    g_arr_m = np.asarray(tab_m["g_noshear"])   # (N,2)
    R_arr_m = np.asarray(tab_m["r11"])         # (N,)

    gamma1_per = (g_arr_p[:,0] - g_arr_m[:,0]) / 2.0
    c_per = (g_arr_p[:,1] + g_arr_m[:,1]) / 2.0

    R1_pair = 0.5 * (R_arr_p + R_arr_m)

    indices = np.arange(N)
    chunks = np.array_split(indices, njac)

    # full values before jackknifing
    shear_est = np.nanmean(gamma1_per)/np.nanmean(R1_pair)
    m_full = shear_est/shear_true - 1
    c_full = np.nanmean(c_per)

    m_jack = []
    c_jack = []
    r11_jack = []
    r11_p_jack = []
    r11_m_jack = []
    for chunk in chunks:
        mask = np.ones(N, dtype=bool)
        mask[chunk] = False
        g_mean_chunk = np.nanmean(gamma1_per[mask])
        R_mean_chunk = np.nanmean(R1_pair[mask])
        shear_chunk = g_mean_chunk/R_mean_chunk
        m_chunk = shear_chunk/shear_true - 1
        c_chunk = np.nanmean(c_per[mask])
        m_jack.append(m_chunk)
        c_jack.append(c_chunk)
        r11_jack.append(R_mean_chunk)
        r11_p_jack.append(np.nanmean(R_arr_p[mask]))
        r11_m_jack.append(np.nanmean(R_arr_m[mask]))

    m_jack = np.array(m_jack)
    c_jack = np.array(c_jack)
    r11_jack = np.array(r11_jack)
    r11_p_jack = np.array(r11_p_jack)
    r11_m_jack = np.array(r11_m_jack)

    m_mean = np.mean(m_jack)
    c_mean = np.mean(c_jack)
    r11_mean = np.mean(r11_jack)
    r11_p_mean = np.mean(r11_p_jack)
    r11_m_mean = np.mean(r11_m_jack)
    m_err = np.sqrt((njac - 1) * np.mean((m_jack - m_mean)**2))
    c_err = np.sqrt((njac - 1) * np.mean((c_jack - c_mean)**2))
    r11_err = np.sqrt((njac-1) * np.mean((r11_jack - r11_mean)**2))
    r11_p_err = np.sqrt((njac-1) * np.mean((r11_p_jack - r11_p_mean)**2))
    r11_m_err = np.sqrt((njac-1) * np.mean((r11_m_jack - r11_m_mean)**2))

    return m_full, c_full, m_mean, m_err, c_mean, c_err, m_jack, c_jack, r11_mean, r11_err, r11_p_mean, r11_p_err, r11_m_mean, r11_m_err

In [3]:
seed = 31415
rng = np.random.RandomState(seed)
shear_true = 0.01
noise = 1e-6

Tguess = 4*0.141**2
ntry = 20
nobs=100
lm_pars = {'maxfev':2000, 'xtol':5.0e-5, 'ftol':5.0e-5}
psf_lm_pars={'maxfev': 4000, 'xtol':5.0e-5,'ftol':5.0e-5}
mcal_pars= {'psf': 'dilate', 'mcal_shear': 0.01}
types = ['noshear', '1p', '1m']
prior = _get_priors(seed)

fitter = ngmix.fitting.Fitter(model="gauss", prior=prior, fit_pars=lm_pars)
guesser = ngmix.guessers.TPSFFluxAndPriorGuesser(rng=rng, T=Tguess, prior=prior)

psf_fitter = ngmix.fitting.Fitter(model='gauss', fit_pars=psf_lm_pars)
psf_guesser = ngmix.guessers.SimplePSFGuesser(rng=rng)

psf_runner = ngmix.runners.PSFRunner(fitter=psf_fitter, guesser=psf_guesser, ntry=ntry)
runner = ngmix.runners.Runner(fitter=fitter, guesser=guesser, ntry=ntry)

psf = mcal_pars['psf']
mcal_shear = mcal_pars['mcal_shear']
boot = ngmix.metacal.MetacalBootstrapper(
    runner=runner, psf_runner=psf_runner,
    rng=rng,
    psf=psf,
    step = mcal_shear,
    types=types,
)


In [4]:
data_list_p = []
data_list_m = []

for i in progress(10, miniters=10):
    obs0, obsp, obsm = make_data(rng=rng, noise=noise, shear_true=shear_true)
    data_p = process_obs(obsp, boot)
    data_m = process_obs(obsm, boot)
    data_list_p.append(data_p)
    data_list_m.append(data_m)

10/10 100%


In [9]:
# --- usage ---
tab_p = shear_data_to_table(data_list_p)
tab_m = shear_data_to_table(data_list_m)

njac = 5
m_full, c_full, m, merr, c, cerr, m_jk, c_jk, r11_mean, r11_err, _, _, _, _ = jackknife_mc_v2(tab_p, tab_m, shear_true, njac=njac)


s2n_p = np.mean(tab_p["s2n_noshear"])
s2n_m = np.mean(tab_m["s2n_noshear"])
s2n = 0.5*(s2n_m + s2n_p)

display(Math(rf"\mathrm{{S/N}} = {s2n:.3f}"))
display(Math(rf"R_{{11}} = {r11_mean:.3f} \pm {r11_err:.3f}"))
exp_m = int(np.floor(np.log10(abs(m)))) if m != 0 else 0
exp_c = int(np.floor(np.log10(abs(c)))) if c != 0 else 0

m_scaled = m / 10**exp_m
merr_scaled = (merr) / 10**exp_m

c_scaled = c / 10**exp_c
cerr_scaled = (cerr) / 10**exp_c

display(Math(rf"m = ({m_scaled:.3f} \pm {merr_scaled:.3f}) \times 10^{{{exp_m}}}"))
display(Math(rf"c = ({c_scaled:.3f} \pm {cerr_scaled:.3f}) \times 10^{{{exp_c}}}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [10]:
tab_p["g_noshear"]

-0.25452845112782957 .. -0.09840532468744165
0.29668976843980466 .. 0.11511352902620424
-0.7485024015063375 .. 0.007952454561778247
0.12742755326559774 .. 0.25432764698909305
0.24224383812708247 .. -0.16894506452844438
-0.020357868020232685 .. 0.0025498894233280758
-0.697713649800487 .. 0.05908239863639384
-0.12753244754536422 .. 0.1495612852319309
0.3060253562050886 .. 0.10201224491949232
-0.46695304029380236 .. -0.09993549718610337
